### Diabetes Prediction - Model Buiding and Evaluation

#### Introduction
This notebook focuses on developing and evaluating different ML classification models to predict the diabetes status.
For that the cleaned dataset will be reloaded and recreated all the preprocessing. This will be used to train multiple classification algorithms.

#### Objectives
- prepare the dataset for model building
- build multiple classification algorithms
- evalution will be conducted based on different metrices and compare the model
- select and save the best performing model

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [3]:
# reload dataset
data = pd.read_csv('../data/diabetes_cleaned.csv')
df3 = data.copy()
df3.head()

,age_group,gender,bmi,waist_circumference_cm,family_history_diabetes,family_history_obesity,family_history_hypertension,physical_activity_level,exercise_days_per_week,sedentary_hours_per_day,...,sleep_hours,stress_level,diet_quality_score,added_sugar_grams_per_day,fiber_grams_per_day,fruit_servings_per_day,systolic_bp,diastolic_bp,triglycerides_mg_dl,diabetes_status
0,31-40,Male,27.25,96.3,No,No,No,Extremely Active,7,6.6,...,6.3,Low,57.0,55.6,23.8,3.5,118.0,67.0,215.0,0.0
1,41-50,Female,31.47,88.5,Yes,No,Yes,Lightly Active,1,8.2,...,8.4,Moderate,51.3,79.3,4.5,1.4,118.0,80.0,203.2,0.0
2,51-60,Male,23.58,93.6,Yes,No,No,Lightly Active,1,5.4,...,5.1,Low,53.1,75.6,9.0,0.0,118.0,81.0,121.6,0.0
3,51-60,Female,23.62,71.3,Yes,No,Yes,Sedentary,1,6.2,...,6.1,High,53.8,30.1,11.5,3.3,120.0,83.0,106.5,1.0
4,51-60,Female,32.51,81.2,Yes,No,Yes,Lightly Active,0,6.2,...,6.7,Moderate,57.2,76.8,17.9,2.3,153.0,76.0,174.8,1.0


In [6]:
df3.shape

(20000, 21)

In [7]:
df3.isnull().sum()

age_group                      0
gender                         0
bmi                            0
waist_circumference_cm         0
family_history_diabetes        0
family_history_obesity         0
family_history_hypertension    0
physical_activity_level        0
exercise_days_per_week         0
sedentary_hours_per_day        0
smoking_status                 0
sleep_hours                    0
stress_level                   0
diet_quality_score             0
added_sugar_grams_per_day      0
fiber_grams_per_day            0
fruit_servings_per_day         0
systolic_bp                    0
diastolic_bp                   0
triglycerides_mg_dl            0
diabetes_status                0
dtype: int64

In [8]:
# separate features and target variable
X = df3.drop('diabetes_status', axis = 1)
y = df3['diabetes_status']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (20000, 20)
Target Shape: (20000,)


In [10]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

In [11]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (16000, 20)
X_test: (4000, 20)
y_train: (16000,)
y_test: (4000,)


In [12]:
# categorizing columns

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
ordinal_features = ['age_group', 'physical_activity_level', 'stress_level']
nominal_features = ['gender', 'smoking_status']
binary_features = ['family_history_diabetes', 'family_history_obesity', 'family_history_hypertension']


In [13]:
print("Numerical columns:", len(num_cols))
print("Ordinal columns:", len(ordinal_features))
print("Nominal columns:", len(nominal_features))
print("Binary columns:", len(binary_features))


Numerical columns: 12
Ordinal columns: 3
Nominal columns: 2
Binary columns: 3


In [14]:
# defining orinal categories
ordinal_categories = [
    ['18-30', '31-40', '41-50', '51-60', '61-70', '71-90'],
    ['Sedentary', 'Lightly Active', 'Moderately Active', 'Very Active', 'Extremely Active'],
    ['Low', 'Moderate', 'High', 'Very High']
]

In [18]:
# defining pipelines

numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

ordinal_pipeline = Pipeline([
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value= -1
    ))
])

nominal_pipeline = Pipeline([
    ('encoder', OneHotEncoder(
        drop = 'first',
        sparse_output = False, 
        handle_unknown='ignore' 
        
    ))
])

binary_pipeline = Pipeline([
    ('encoder', OrdinalEncoder(
        categories = [['No', 'Yes']] * 3 
    ))
    
])

In [19]:
# combining transformations using columntranformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, num_cols), # the form of list is (name, transformer, columns)
        ('ordinal', ordinal_pipeline, ordinal_features),
        ('nominal', nominal_pipeline, nominal_features),
        ('binary', binary_pipeline, binary_features)
    ]
)

Here we are going to combine the preprocessing steps and LR classifier into a single pipeline.